# A sliding-window dataset with `Tensor.unfold`


In [1]:
import torch
from torch.utils.data import DataLoader, Dataset


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


class GPTDatasetStrided(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        token_ids = torch.tensor(
            tokenizer.encode(txt, allowed_special={"<|endoftext|>"}),
            dtype=torch.long,
        )
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        windows = token_ids.unfold(0, max_length + 1, stride)
        self.input_view = windows[..., :-1]
        self.target_view = windows[..., 1:]

    def __len__(self):
        return self.input_view.size(0)

    def __getitem__(self, idx):
        return self.input_view[idx], self.target_view[idx]

## Checking the input and target pairs

- Let's compare every example from both datasets using *The Verdict* and the Chapter 2 defaults for `max_length` and `stride`
- The `GPTDatasetV1` code above comes from the [Chapter 2 notebook](../01_main-chapter-code/ch02.ipynb)
- The `GPTDatasetStrided` code is a more efficient alternative (although a tad harder to read unless you are an advanced PyTorch user)

In [2]:
from pathlib import Path

import tiktoken

text = Path("../01_main-chapter-code/the-verdict.txt").read_text(encoding="utf-8")
tokenizer = tiktoken.get_encoding("gpt2")
max_length = 256
stride = 128

chapter = GPTDatasetV1(text, tokenizer, max_length, stride)
strided = GPTDatasetStrided(text, tokenizer, max_length, stride)
assert len(chapter) == len(strided)
for (x, y), (x_view, y_view) in zip(chapter, strided):
    assert torch.equal(x, x_view)
    assert torch.equal(y, y_view)

print(f"All {len(chapter)} input and target pairs match.")

All 39 input and target pairs match.


## Timing setup and one pass through the data


In [3]:
from statistics import median
from time import perf_counter


def benchmark(dataset_class):
    setup_times = []
    epoch_times = []
    for _ in range(31):
        start = perf_counter()
        dataset = dataset_class(text, tokenizer, max_length, stride)
        setup_times.append((perf_counter() - start) * 1000)

        loader = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=0)
        start = perf_counter()
        for _batch in loader:
            pass
        epoch_times.append((perf_counter() - start) * 1000)

    return median(setup_times[1:]), median(epoch_times[1:])


print("Dataset             Setup (ms)   One pass (ms)")
for dataset_class in (GPTDatasetV1, GPTDatasetStrided):
    setup_ms, epoch_ms = benchmark(dataset_class)
    print(f"{dataset_class.__name__:18} {setup_ms:10.2f} {epoch_ms:15.2f}")

Dataset             Setup (ms)   One pass (ms)
GPTDatasetV1             1.92            0.15
GPTDatasetStrided        1.06            0.17


## How much memory do the token IDs use?

- Each 64-bit token ID takes 8 bytes
- The chapter dataset keeps `max_length` input IDs and `max_length` target IDs for every window
- The strided dataset keeps one tensor for the whole text, and each window points into that storage

In [4]:
def token_storage_kib(dataset):
    storage_sizes = {}
    for inputs, targets in dataset:
        for tensor in (inputs, targets):
            storage = tensor.untyped_storage()
            storage_sizes[storage.data_ptr()] = storage.nbytes()
    return sum(storage_sizes.values()) / 1024


print("Stride  Windows   Chapter (KiB)   Strided (KiB)")
for step in (stride, 1):
    chapter_data = GPTDatasetV1(text, tokenizer, max_length, step)
    strided_data = GPTDatasetStrided(text, tokenizer, max_length, step)
    chapter_kib = token_storage_kib(chapter_data)
    strided_kib = token_storage_kib(strided_data)
    print(f"{step:6} {len(chapter_data):8,} {chapter_kib:14,.1f} {strided_kib:15,.1f}")

Stride  Windows   Chapter (KiB)   Strided (KiB)
   128       39          156.0            40.2


     1    4,889       19,556.0            40.2
